# Landmark TSFEL Feature Extraction

Build the temporal-domain landmark feature table by running TSFEL over the raw mediapipe landmark CSVs in `HANDS_LANDMARKS/` and `BODY_LANDMARKS/`. The filename convention is `{sign}_{subject}_{rep}.csv`. Outputs `general_db.csv` and `feature_names.txt`, both consumed by `feature_extraction_chronos_lm.ipynb` and `landmarks/scripts/grid_search.py`.

## Configuration

In [ ]:
LEFT_HAND_DIR = "HANDS_LANDMARKS/LEFT_HAND_LANDMARKS"
RIGHT_HAND_DIR = "HANDS_LANDMARKS/RIGHT_HAND_LANDMARKS"
BODY_DIR = "BODY_LANDMARKS"
OUTPUT_CSV = "general_db.csv"
FEATURE_NAMES_TXT = "feature_names.txt"
SAMPLE_RATE_HZ = 60


## Setup and TSFEL config

In [ ]:
import pandas as pd
import tsfel
import numpy as np
import os, glob
from tqdm import tqdm


files = glob.glob(os.path.join(LEFT_HAND_DIR, "*.csv"))

just_route_files = [os.path.split(f)[-1] for f in files]
config = tsfel.get_features_by_domain("temporal")

def feature_extractor(X:pd.DataFrame, fs=SAMPLE_RATE_HZ):
    features = tsfel.time_series_features_extractor(timeseries=X, fs=fs, config=config,verbose=0)
    return features

## Determine common frame length

In [ ]:
max_frames = min([len(pd.read_csv(os.path.join(LEFT_HAND_DIR, i))) for i in just_route_files])


In [ ]:
max_frames

## Build the consolidated feature table

In [ ]:

general_db = None

for route in tqdm(just_route_files):

    left_hand_file = pd.read_csv(os.path.join(LEFT_HAND_DIR, route))
    right_hand_file = pd.read_csv(os.path.join(RIGHT_HAND_DIR, route))
    body_file = pd.read_csv(os.path.join(BODY_DIR, route))

    info = route.split(".")[0]
    
    sign, subject, rep = info.split("_")

    left_hand_drop = left_hand_file.drop(columns=["Unnamed: 0"] + [i for i in left_hand_file.columns if "z" in i],axis=1)
    right_hand_drop = right_hand_file.drop(columns=["Unnamed: 0"] + [i for i in right_hand_file.columns if "z" in i],axis=1)
    body_drop = body_file.drop(columns=["Unnamed: 0"] + [i for i in body_file.columns if "z" in i or any([str(j) in i for j in range(23,33)])],axis=1)


    left_hand_drop.columns = [f"{col}_LH" for col in left_hand_drop.columns]
    right_hand_drop.columns = [f"{col}_RH" for col in right_hand_drop.columns]
    body_drop.columns = [f"{col}_B" for col in body_drop.columns]

    left_hand_drop = left_hand_drop.reindex(range(max_frames)).interpolate()
    right_hand_drop = right_hand_drop.reindex(range(max_frames)).interpolate()
    body_drop = body_drop.reindex(range(max_frames)).interpolate()

    general_information = pd.concat([left_hand_drop, right_hand_drop, body_drop], axis=1)
    general_information["label"] = int(sign)
    general_information["subject"] = int(subject)
    general_information["rep"] = int(rep)

    if general_db is None:
        general_db = general_information
    else:
        general_db = pd.concat([general_db, general_information], axis=0)


    #features_information = feature_extractor(general_information)

    
    


    


## Persist outputs

In [ ]:
with open(FEATURE_NAMES_TXT,"w") as f:
    for column in general_db.columns:
        f.write(f"{column}\n")
    f.close()


general_db.to_csv(OUTPUT_CSV,encoding="utf-8", index=False)